In [ ]:

# ---- Query DBpedia SPARQL endpoint ----
def query_dbpedia(sparql_query):
    """
    Sends a SPARQL query to the DBpedia endpoint and returns the JSON result.
    Returns None if the request fails.
    """
    endpoint = "https://dbpedia.data.dice-research.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    params = {"query": sparql_query}
    try:
        response = requests.get(endpoint, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException:
        return None

In [ ]:
# ---- Load queries from CSV ----
df = pd.read_csv("combined_unique_select_queries.csv")
print(f"Total queries loaded: {len(df)}")

# ---- Execute queries and collect successful ones ----
counter = 0
failed = 0
successful_queries = []

for index, row in df.iterrows():
    sparql_query = row.get("text")
    if sparql_query:
        results = query_dbpedia(sparql_query)
        if results and "results" in results and results["results"]["bindings"]:
            successful_queries.append({
                "sparql_query": sparql_query,
                "results": results["results"]["bindings"]
            })
            counter += 1
        else:
            failed += 1

print(f"Successful queries: {len(successful_queries)}")
print(f"Failed queries: {failed}")

In [ ]:

# ---- Save successful queries to JSON ----
with open("successful_queries.json", "w", encoding="utf-8") as f:
    json.dump(successful_queries, f, ensure_ascii=False, indent=2)

# ---- Load successful queries ----
with open("successful_queries.json", "r") as f:
    data = json.load(f)

# ---- Extract result strings for similarity comparison ----
def extract_result_strings(entry):
    """
    Extracts unique result strings from a query result entry.
    """
    values = set()
    for result in entry.get("results", []):
        for k, v in result.items():
            if isinstance(v, dict) and 'value' in v:
                values.add(f"{k}:{v['value']}")
    return values


# ---- Preprocess all result sets ----
preprocessed = [(i, entry['sparql_query'], extract_result_strings(entry)) for i, entry in enumerate(data)]


In [ ]:
# ---- Worker function for multiprocessing ----
def process_query(args):
    """
    Finds similar queries based on Jaccard similarity of result sets.
    Returns a dictionary with base query and its similar queries.
    """
    base_id, base_query, base_results = args
    local_similars = []
    for sim_id, sim_query, sim_results in preprocessed[base_id + 1:]:
        intersection = base_results & sim_results
        union = base_results | sim_results
        jaccard_score = len(intersection) / len(union) if union else 0
        if jaccard_score > 0.8:
            local_similars.append({
                "similar_query": sim_query.strip(),
                "jaccard_score": round(jaccard_score, 3),
                "similar_results": list(sim_results)
            })
            if len(local_similars) >= 5:
                break
    return {
        "base_query": base_query.strip(),
        "base_results": list(base_results),
        "similar_queries": local_similars
    } if local_similars else None

In [ ]:
# ---- Multiprocessing to compute similarities ----
cpu_count = multiprocessing.cpu_count()
args_list = [(i, q, r) for i, (idx, q, r) in enumerate(preprocessed)]
results = []
with multiprocessing.Pool(cpu_count) as pool:
    for result in tqdm(pool.imap_unordered(process_query, args_list), total=len(args_list)):
        if result:
            results.append(result)

# ---- Save similarity results to JSON ----
with open("jaccard_similarities_optimized.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("✅ JSON file written: jaccard_similarities_optimized.json")